# Project 2 - Traffic and Air Quality in New York City
### Exploring whether busier roads mean dirtier air

In this project, I’m using two datasets from NYC Open Data to explore a simple but meaningful question: *How does traffic volume relate to air quality levels across New York City?*

Air pollution, especially fine particulate matter (PM2.5), is a major urban health concern. At the same time, NYC streets carry enormous daily traffic, and vehicles are a major source of pollution. It feels intuitive that more cars might lead to worse air quality—but I want to see whether the data actually supports that assumption.

**Research Question:**  
*How does traffic volume relate to air quality levels across New York City?*

**Hypothesis:**  
Higher traffic volume is associated with worse air quality.

To investigate this, I’ll work with two datasets:
1. **NYC Air Quality Data** – includes measures like PM2.5 and nitrogen dioxide across neighborhoods and time periods.
2. **NYC Traffic Volume Counts** – hourly counts of vehicles on specific roadway segments, from which I’ll derive daily and monthly traffic trends.

These two datasets cover different aspects—pollution and mobility—but both share a common time column, which will allow me to merge them and visualize their relationship in a single chart.

### Data Sources

Both datasets used in this project come from **NYC Open Data**, the city’s open data portal.

- **Air Quality Data (PM2.5, NO2, Ozone, etc.)**  
  Source: NYC Environment & Health Data Portal  
  https://data.cityofnewyork.us/Environment/Air-Quality/c3uy-2p5r

- **NYC Traffic Volume Counts (Hourly Traffic Data)**  
  Source: NYC Department of Transportation (NYC DOT)  
  https://data.cityofnewyork.us/Transportation/Traffic-Volume-Counts/btm5-ppia/about_data

I downloaded both CSV files directly from NYC Open Data and work with the cleaned versions locally in this notebook.

## Dataset A: Air Quality

The air quality dataset comes from NYC Open Data. It includes measurements of several pollutants—such as PM2.5, nitrogen dioxide (NO2), and ozone (O3)—across different geographic areas in New York City.

Each row represents a pollutant measured during a specific time period (for example, “Summer 2023”), along with:
- the pollutant name,
- the measurement unit,
- the location (UHF or community district),
- the time period,
- the numerical value of the measurement.

For this project, I focus on **fine particulate matter (PM2.5)**. PM2.5 is widely used as a key indicator of air pollution because the particles are small enough to enter the bloodstream and are strongly associated with health risks such as asthma, cardiovascular illness, and premature mortality.

Since my research question examines whether busier roads are associated with worse air quality, PM2.5 is the most relevant pollutant to analyze.

#### Loading and Previewing the Dataset

In [ ]:
import pandas as pd

# Load Dataset A: Air Quality
aq = pd.read_csv("Air_Quality_20251121.csv")

# Preview the data
aq.head()

,Unique ID,Indicator ID,Name,Measure,Measure Info,Geo Type Name,Geo Join ID,Geo Place Name,Time Period,Start_Date,Data Value,Message
0,878218,386,Ozone (O3),Mean,ppb,UHF42,402,West Queens,Summer 2023,06/01/2023,34.365989,NaN
1,876975,375,Nitrogen dioxide (NO2),Mean,ppb,UHF42,501,Port Richmond,Summer 2023,06/01/2023,11.331992,NaN
2,876900,375,Nitrogen dioxide (NO2),Mean,ppb,UHF42,207,East Flatbush - Flatbush,Summer 2023,06/01/2023,12.020333,NaN
3,877140,375,Nitrogen dioxide (NO2),Mean,ppb,CD,205,Fordham and University Heights (CD5),Summer 2023,06/01/2023,14.123178,NaN
4,874556,365,Fine particles (PM 2.5),Mean,mcg/m3,UHF34,410,Rockaways,Summer 2023,06/01/2023,8.150637,NaN


This preview shows the pollutant type (Name), the geographic identifier (Geo Join ID and Geo Place Name), the time period, and the numerical pollutant value (Data Value).

#### Cleaning and Preparing the Air Quality Data

To merge this dataset with traffic data later, I first convert the Start_Date column into a usable datetime format and extract the month. This creates a consistent time variable across both datasets.

Next, because the dataset includes many pollutants, I filter it to keep only rows where Name == "Fine particles (PM 2.5)". This isolates PM2.5 values for each geographic area.

Finally, to make the dataset compatible with monthly traffic trends, I compute the average PM2.5 value for each month across NYC.

In [ ]:
# Convert Start_Date to datetime
aq["Start_Date"] = pd.to_datetime(aq["Start_Date"], errors="coerce")

# Extract month
aq["month"] = aq["Start_Date"].dt.month

# Filter for PM2.5 pollutant only
aq_pm25 = aq[aq["Name"] == "Fine particles (PM 2.5)"].copy()

# Compute monthly average PM2.5 across NYC
pm25_monthly = aq_pm25.groupby("month")["Data Value"].mean().reset_index()

# Rename for clarity
pm25_monthly.rename(columns={"Data Value": "pm25"}, inplace=True)

pm25_monthly

,month,pm25
0,1,7.147393
1,6,9.427661
2,12,9.475098


This produces a clean dataset with two columns—month and pm25—which will allow me to merge it with the traffic dataset and visualize their relationship.

## Dataset B: Traffic Volume Counts
The second dataset comes from NYC Open Data, published by the NYC Department of Transportation. It contains hourly vehicle counts on specific roadway segments across New York City. Each row corresponds to one traffic count session for a particular street segment on a specific date.
This dataset provides insight into road usage, congestion, and mobility patterns, which allows me to compare daily traffic activity with air quality trends from Dataset A.

The dataset includes:
- the roadway name and cross streets,
- the direction of traffic flow,
- the date of the observation,
- and **24 hourly traffic count columns** (e.g., “12:00–1:00 AM”, “1:00–2:00 AM”, …, “11:00–12:00 PM”).

Altogether, these hourly counts reflect how busy each roadway segment was during a given day.

Because my research question focuses on the relationship between traffic volume and air quality, this dataset provides a quantitative measure of daily mobility and congestion across the city.


#### Loading and Previewing the Dataset

In [ ]:
# Load Dataset B: Traffic Volume Counts
traffic = pd.read_csv("Traffic_Volume_Counts_20251121.csv")

# Preview
traffic.head()

,ID,SegmentID,Roadway Name,From,To,Direction,Date,12:00-1:00 AM,1:00-2:00AM,2:00-3:00AM,...,2:00-3:00PM,3:00-4:00PM,4:00-5:00PM,5:00-6:00PM,6:00-7:00PM,7:00-8:00PM,8:00-9:00PM,9:00-10:00PM,10:00-11:00PM,11:00-12:00AM
0,1,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,01/09/2012,20,10,11,...,104,105,147,120,91,83,74,49,42,42
1,2,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,01/10/2012,21,16,8,...,102,98,133,131,95,73,70,63,42,35
2,3,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,01/11/2012,27,14,6,...,115,115,130,143,106,89,68,64,56,43
3,4,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,01/12/2012,22,7,7,...,71,127,122,144,122,76,64,58,64,43
4,5,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,01/13/2012,31,17,7,...,113,126,133,135,102,106,58,58,55,54


From the preview, we can see the hourly columns and the Date field, which will be essential for aligning the traffic dataset with the air quality dataset.

#### Cleaning and Preparing the Traffic Data
To compare traffic with air quality, I need a single daily traffic value per observation.
The dataset provides 24 hourly counts, so I will:
- identify the 24 hourly columns
- convert them to numeric (some may be strings)
- sum across the row to compute total daily traffic volume


##### 1. Identify all hourly traffic columns
Hourly columns contain ":" in their names, so I extract them programmatically:

In [ ]:
# Identify all hourly traffic count columns automatically
hour_cols = [col for col in traffic.columns if ":" in col]
hour_cols[:5]  # show a preview

['12:00-1:00 AM', '1:00-2:00AM', '2:00-3:00AM', '3:00-4:00AM', '4:00-5:00AM']

##### 2. Convert hourly columns to numeric
Some hourly values are stored as strings, so I convert them safely to numeric:

In [ ]:
# Convert hourly columns to numeric values
traffic[hour_cols] = traffic[hour_cols].apply(pd.to_numeric, errors="coerce")

##### 3. Compute total daily traffic volume
Now that hourly traffic is numeric, I can sum across all 24 hours:

In [ ]:
# Compute total daily traffic volume
traffic["daily_volume"] = traffic[hour_cols].sum(axis=1)

# Preview
traffic[["Date", "daily_volume"]].head()

,Date,daily_volume
0,2012-01-09,1529.0
1,2012-01-10,1424.0
2,2012-01-11,1574.0
3,2012-01-12,1559.0
4,2012-01-13,1659.0


##### 4. Convert “Date” to datetime and extract month
The traffic dataset provides a date for each count session.
To compare traffic with monthly air quality, I need to extract the **month** from the Date column.

In [ ]:
# Convert Date column to datetime
traffic["Date"] = pd.to_datetime(traffic["Date"], errors="coerce")

# Extract month as a number
traffic["month"] = traffic["Date"].dt.month

traffic[["Date", "month", "daily_volume"]].head()

,Date,month,daily_volume
0,2012-01-09,1,1529.0
1,2012-01-10,1,1424.0
2,2012-01-11,1,1574.0
3,2012-01-12,1,1559.0
4,2012-01-13,1,1659.0


##### 5. Compute monthly average daily traffic
Now that each row has a month and a daily traffic volume, I can compute the average daily traffic per month across the dataset.

In [ ]:
# Compute monthly average traffic volume
traffic_monthly = traffic.groupby("month")["daily_volume"].mean().reset_index()

# Rename for clarity
traffic_monthly.rename(columns={"daily_volume": "avg_daily_traffic"}, inplace=True)

traffic_monthly

,month,avg_daily_traffic
0,1,5087.543328
1,2,4893.392774
2,3,6150.011876
3,4,5563.535127
4,5,4725.341564
5,9,6168.311246
6,10,6175.465673
7,11,5859.671381
8,12,6426.470588


## Merging Air Quality and Traffic Datasets
Now that I have monthly averages for both PM2.5 (from Dataset A) and traffic volume (from Dataset B), the next step is to combine them into a single dataset. This will allow me to directly compare how the two variables move across months and visualize their relationship in one chart, as required for the project.

Both datasets contain a month column, which I will use as the joining key.


#### Merge the two datasets on "month"

In [ ]:
# Merge Dataset A (pm25_monthly) and Dataset B (traffic_monthly)
merged = pd.merge(pm25_monthly, traffic_monthly, on="month", how="inner")

merged

,month,pm25,avg_daily_traffic
0,1,7.147393,5087.543328
1,12,9.475098,6426.470588


This merged table now includes:
	•	the month number
	•	the average PM2.5 level for that month
	•	the average daily traffic volume for that month

This combined dataset will be used to create a single visualization showing how traffic and air quality relate to each other across months.

## Visualizing the Relationship Between Traffic Volume and PM2.5
With both datasets merged into a single monthly table, I can now create one visualization that shows how air quality and traffic volume move together across the year. Since the two variables are measured on very different scales, I will use a dual-axis chart:
- Left y-axis: average daily traffic
- Right y-axis: PM2.5 concentration (µg/m³)

This allows both lines to be visible on the same chart without distorting the trends.

#### Plot traffic volume and PM2.5 on the same chart

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# Bar chart for traffic volume
fig.add_trace(
    go.Bar(
        x=merged["month"],
        y=merged["avg_daily_traffic"],
        name="Avg Daily Traffic",
        marker_color="steelblue",
        yaxis="y1",
    )
)

# Line chart for PM2.5
fig.add_trace(
    go.Scatter(
        x=merged["month"],
        y=merged["pm25"],
        name="PM2.5 (µg/m³)",
        mode="lines+markers",
        marker_color="firebrick",
        yaxis="y2",
    )
)

# Update layout
fig.update_layout(
    title="Monthly Traffic Volume vs PM2.5 Levels in New York City",
    xaxis_title="Month",
    yaxis=dict(title="Average Daily Traffic", side="left"),
    yaxis2=dict(title="PM2.5 (µg/m³)", overlaying="y", side="right"),
    barmode="group",
    height=450,
    legend=dict(title="Measures"),
)

fig.show()